# TF-IDF (Term Frequency - Inverse Document Frequency)

El concepto de **TF-IDF** es una de las técnicas más elegantes y fundamentales en el Procesamiento de Lenguaje Natural (NLP). 

Su objetivo principal es responder a una pregunta sencilla: **¿Qué tan importante es una palabra en un documento específico, teniendo en cuenta todo un conjunto de documentos?**

A continuación, se detalla la explicación conceptual, un ejemplo manual (Toy Example) y cómo implementarlo en Python.

## 1. La Teoría: ¿Por qué TF-IDF?

Si queremos saber de qué trata un texto, podríamos simplemente contar qué palabras aparecen más veces. Sin embargo, esto tiene un problema: palabras como "el", "la", "de" o "y" aparecerán muchísimas veces, pero no nos dicen nada sobre el significado real del texto.

TF-IDF resuelve esto equilibrando dos fuerzas:

*   **TF (Frecuencia del Término):** Premia las palabras que aparecen *muchas* veces en un documento específico.
*   **IDF (Frecuencia Inversa de Documento):** Penaliza las palabras que aparecen en *todos* los documentos (porque si una palabra está en todas partes, no es útil para distinguir o caracterizar un documento en particular).

### Las Fórmulas Matemáticas

**1. Term Frequency (TF)**
Mide la frecuencia de un término $t$ en un documento $d$. Hay varias formas de calcularlo, pero la más intuitiva es la frecuencia bruta dividida por el total de palabras de ese documento:

$$TF(t, d) = \frac{\text{Veces que aparece } t \text{ en } d}{\text{Total de palabras en } d}$$

**2. Inverse Document Frequency (IDF)**
Mide qué tanta información aporta el término $t$ en todo el corpus (conjunto) de documentos $D$ (donde $N$ es el número total de documentos). Se usa un logaritmo para amortiguar el efecto de palabras muy raras:

$$IDF(t, D) = \log\left(\frac{N}{\text{Documentos que contienen } t}\right)$$

**3. TF-IDF**
Finalmente, multiplicamos ambos valores. Si una palabra aparece mucho en un documento (TF alto) pero poco en el resto de documentos (IDF alto), su puntuación TF-IDF se dispara.

$$TF\text{-}IDF(t, d, D) = TF(t, d) \times IDF(t, D)$$

## 2. Toy Example (Cálculo Manual)

Imagina que tenemos un corpus de solo 3 documentos (frases cortas) con un total de N = 3:

- **Doc A:** "el gato corre" (3 palabras)
- **Doc B:** "el perro ladra" (3 palabras)
- **Doc C:** "el gato duerme" (3 palabras)

Vamos a calcular el TF-IDF para las palabras del **Doc A ("el gato corre")** usando logaritmo en base 10 para simplificar los cálculos:

| Término | TF en Doc A | Cálculo del IDF | Resultado IDF | TF-IDF (TF × IDF) |
|---|---|---|---|---|
| **el** | 1/3 = 0.333 | $\log(3/3) = \log(1)$ | 0 | **0** |
| **gato** | 1/3 = 0.333 | $\log(3/2)$ | 0.176 | **0.058** |
| **corre** | 1/3 = 0.333 | $\log(3/1)$ | 0.477 | **0.159** |

**Conclusión del ejemplo:** 
* La palabra "el" obtiene un TF-IDF de **0**. La matemática ha eliminado automáticamente una palabra vacía sin necesidad de una lista manual.
* La palabra "corre" obtiene el puntaje más alto (**0.159**) porque es única de ese documento; es la palabra que mejor define al Doc A frente a los demás.

## 3. Implementación en Python

En el mundo real, no calculamos esto a mano. Usamos `scikit-learn`, que es el estándar de la industria. 

> **Nota técnica:** `scikit-learn` aplica por defecto un par de trucos matemáticos extra: usa logaritmo natural ($\ln$), añade un "+1" a los denominadores para evitar divisiones por cero (suavizado o *smoothing*), y normaliza los vectores finales para que la longitud de cada documento no afecte injustamente.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# 1. Nuestro corpus de documentos
documentos = [
    "el gato corre",
    "el perro ladra",
    "el gato duerme"
]

# 2. Inicializamos el modelo
# (Por defecto scikit-learn convierte todo a minúsculas y elimina puntuación)
vectorizer = TfidfVectorizer()

# 3. Calculamos la matriz TF-IDF
tfidf_matrix = vectorizer.fit_transform(documentos)

# 4. Extraemos los nombres de las palabras (vocabulario)
nombres_caracteristicas = vectorizer.get_feature_names_out()

# 5. Lo convertimos en un DataFrame de Pandas para verlo como una tabla
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=nombres_caracteristicas)
df_tfidf.index = ['Doc A', 'Doc B', 'Doc C']

print(df_tfidf)

          corre    duerme        el      gato     ladra     perro
Doc A  0.720333  0.000000  0.425441  0.547832  0.000000  0.000000
Doc B  0.000000  0.000000  0.385372  0.000000  0.652491  0.652491
Doc C  0.000000  0.720333  0.425441  0.547832  0.000000  0.000000
